In [37]:
# import libraries
import pandas as pd
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# load the data
all_posts_df = pd.read_csv("../01_data/clean_data/all_posts_cleaned.csv")

In [38]:
# load cardiff nlp roberta model
model = AutoModelForSequenceClassification.from_pretrained('cardiffnlp/xlm-roberta-base-tweet-sentiment-de')
tokenizer= AutoTokenizer.from_pretrained('cardiffnlp/xlm-roberta-base-tweet-sentiment-de')

In [39]:
# create function to label single post
def label_post(post, model, tokenizer):

    # get token ids and run them through the model
    input_ids = tokenizer(post, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**input_ids)

    # turn logits to probabilities and extract probs for positive and negative class
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    positive_prob = probs[0, 2].item()
    negative_prob = probs[0, 0].item()

    # take the log of the ratio (log-odds) with a smoothing factor
    alpha = 1e-6
    log_ratio = np.log((positive_prob + alpha) / (negative_prob + alpha))

    # get also the predicted class
    sentiment_classes = ["negative", "neutral", "positive"]
    predicted_class = sentiment_classes[probs.argmax()]

    return log_ratio, predicted_class

# apply the function to the full df
all_posts_df[['sentiment_score', 'sentiment_class']] = all_posts_df['text'].apply(
    lambda x: pd.Series(label_post(x, model, tokenizer))
)
# show first few rows
all_posts_df.head()

,id,full_name,birthdate,gender,faction,faction_start,faction_end,profession,district_number,district_name,mandate,clean_handle,text,date,sentiment_score,sentiment_class
0,11004906,Dr. Marie-Agnes Strack-Zimmermann,1958-03-10,weiblich,FDP,2021-10-26,2024-07-15,"selbst. Verlagsrepräsentantin, Publizistin",106.0,Düsseldorf I,Landesliste,masz.bsky.social‬,„Trotz auch harter Debatten gilt mein Dank Min...,2024-06-01 04:39:03.862,3.772661,positive
1,11004278,Matthias Gastel,1970-12-26,männlich,Grünen,2021-10-26,2025-03-25,NaN,262.0,Nürtingen,Landesliste,matthias-gastel.de,War zu Bahn und ÖPNV in Nürnberg. Es ging um d...,2024-06-01 05:52:16.490,-0.200295,neutral
2,11005162,Sascha Müller,1970-04-24,männlich,Grünen,2021-10-26,2025-03-25,Sportjournalist,245.0,Nürnberg-Süd,Landesliste,saschamk.bsky.social,"DGB-Chefin Yasmin Fahimi liegt falsch, wenn si...",2024-06-01 08:02:58.941,-5.528440,negative
3,11004255,Dr. Franziska Brantner,1979-08-24,weiblich,Grünen,2021-10-26,2025-03-25,Sozialwissenschaftlerin,274.0,Heidelberg,Direktwahl,franziskabrantner.de‬,Die deutsche Abhängigkeit von russischem Gas k...,2024-06-01 08:05:43.770,-5.810579,negative
4,11004117,Beate Müller-Gemmeke,1960-10-07,weiblich,Grünen,2021-10-26,2025-03-25,Dipl. Sozialpädagogin (FH),289.0,Reutlingen,Landesliste,gruenebeate.bsky.social,Anders als beim Fußball gibt es in der #Prosti...,2024-06-01 08:55:25.819,-2.312032,neutral


In [46]:
# derive a continuous week index directly
all_posts_df['date'] = pd.to_datetime(all_posts_df['date'])
all_posts_df['week_start'] = all_posts_df['date'].dt.to_period('W').apply(lambda r: r.start_time)

# now group by MP and that week_start (or derive week_number / week_index here)
weekly_sentiment = all_posts_df.groupby(['full_name', 'week_start']).agg(
        mean_sentiment_score=('sentiment_score','mean'),
        n_posts=('sentiment_score','size')).reset_index()

# get a continuous week index
weekly_sentiment['week_index'] = weekly_sentiment['week_start'].rank(method='dense').astype(int) - 1

# for each MP get their first data per week (faction, mandate and districts can change over time)
mp_week_info = all_posts_df.groupby(['full_name','week_start']).agg(
    birthdate=('birthdate','first'),
    gender=('gender','first'),
    faction=('faction','first'),
    profession=('profession','first'),
    district_number=('district_number','first'),
    district_name=('district_name','first'),
    mandate=('mandate','first'),
    clean_handle=('clean_handle','first')
).reset_index()

# join with the original df to get the additional MP information
weekly_sentiment = pd.merge(weekly_sentiment, mp_week_info, how="left", on=["full_name", "week_start"])

# add an age column
weekly_sentiment['birthdate'] = pd.to_datetime(weekly_sentiment['birthdate']) # convert birthdate to a datetime object
weekly_sentiment['age'] = weekly_sentiment['week_start'].dt.year - weekly_sentiment['birthdate'].dt.year # take the naive difference in years (for simplicity)

# show the first rows to see everything worked
weekly_sentiment.head()

,full_name,week_start,mean_sentiment_score,n_posts,week_index,birthdate,gender,faction,profession,district_number,district_name,mandate,clean_handle,age
0,Alexander Müller,2024-08-12,0.013284,1,11,1969-07-17,männlich,FDP,"Dipl.-Informatiker, Berufspilot, Oberstleutnan...",178.0,Rheingau-Taunus - Limburg,Landesliste,alexandermueller.bsky.social,55
1,Alexander Müller,2024-08-19,-2.498072,1,12,1969-07-17,männlich,FDP,"Dipl.-Informatiker, Berufspilot, Oberstleutnan...",178.0,Rheingau-Taunus - Limburg,Landesliste,alexandermueller.bsky.social,55
2,Alexander Müller,2024-08-26,4.603703,1,13,1969-07-17,männlich,FDP,"Dipl.-Informatiker, Berufspilot, Oberstleutnan...",178.0,Rheingau-Taunus - Limburg,Landesliste,alexandermueller.bsky.social,55
3,Alexander Müller,2024-09-30,-3.211948,1,18,1969-07-17,männlich,FDP,"Dipl.-Informatiker, Berufspilot, Oberstleutnan...",178.0,Rheingau-Taunus - Limburg,Landesliste,alexandermueller.bsky.social,55
4,Alexander Müller,2024-11-11,-3.956357,1,24,1969-07-17,männlich,FDP,"Dipl.-Informatiker, Berufspilot, Oberstleutnan...",178.0,Rheingau-Taunus - Limburg,Landesliste,alexandermueller.bsky.social,55


In [47]:
# export the labelled data
all_posts_df.to_csv("../01_data/clean_labelled_data/all_posts_cleaned_labelled.csv")

# export the grouped and labelled data
weekly_sentiment.to_csv("../01_data/clean_labelled_data/weekly_sentiment_scores.csv")